In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================

DATA_DIR = Path("../../dataset")
INPUT_FILE = DATA_DIR / "tiki_products.csv"
OUTPUT_DIR = DATA_DIR / "cleaned"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Kích thước dữ liệu ban đầu:", df.shape)

# Kiểm tra các giá trị bị thiếu
print("Số lượng bị thiếu:",
      df["quantity_sold"].isna().sum())

print("\nKiểu dữ liệu:")
print(df["quantity_sold"].dtype)

# Xem một số dòng bị thiếu số lượng đã bán
cols = [
    col for col in [
        "product_id", "name", "price",
        "quantity_sold", "rating_average",
        "review_count", "url"
    ]
    if col in df.columns
]

display(df.loc[df["quantity_sold"].isna(), cols].head(20))

# Xem các giá trị khác nhau đang tồn tại
print("\nMột số giá trị quantity_sold:")
print(df["quantity_sold"].value_counts(dropna=False).head(20))
# ==========================================
# 2. CHUẨN HÓA TÊN CỘT
# ==========================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\-]+", "_", regex=True)
)

# Chuẩn hóa khoảng trắng ở các cột văn bản
text_cols = df.select_dtypes(include=["object"]).columns

for col in text_cols:
    df[col] = df[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Đã chuẩn hóa tên cột.")


# ==========================================
# 3. LOẠI BỎ CÁC CỘT KHÔNG HỮU ÍCH
# ==========================================

drop_cols = [
    "seller_name",       # Thiếu 100%
    "inventory_status",  # Thiếu 100%
    "order_count",       # Toàn bộ bằng 0
    "favourite_count",   # Toàn bộ bằng 0
    "has_installment",   # Toàn bộ False
    "has_gift",          # Toàn bộ False
    "availability"       # Toàn bộ bằng 1
]

# Chỉ xóa các cột thực sự tồn tại
drop_cols = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=drop_cols)

print("Các cột đã loại bỏ:", drop_cols)
print("Số cột còn lại:", len(df.columns))


# ==========================================
# 4. XỬ LÝ DỮ LIỆU TRÙNG LẶP
# ==========================================

before = len(df)

# Xóa bản ghi trùng hoàn toàn
df = df.drop_duplicates()

print("Số dòng trùng hoàn toàn đã xóa:", before - len(df))

# Kiểm tra ID trùng, không tự ý xóa sản phẩm
if "product_id" in df.columns:
    print(
        "Số product_id trùng:",
        df["product_id"].duplicated().sum()
    )

if "sku" in df.columns:
    print(
        "Số SKU trùng:",
        df["sku"].duplicated().sum()
    )


# ==========================================
# 5. CHUẨN HÓA KIỂU DỮ LIỆU
# ==========================================

numeric_cols = [
    "product_id",
    "sku",
    "seller_id",
    "price",
    "original_price",
    "discount",
    "discount_rate",
    "rating_average",
    "review_count",
    "quantity_sold",
    "category_id"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Chuyển thời gian crawl sang datetime
if "crawled_at" in df.columns:
    df["crawled_at"] = pd.to_datetime(
        df["crawled_at"], errors="coerce", utc=True
    )


# ==========================================
# 6. KIỂM TRA GIÁ VÀ TỶ LỆ GIẢM GIÁ
# ==========================================

# Tính lại mức giảm giá từ giá gốc và giá bán
valid_price = (
    df["price"].notna()
    & df["original_price"].notna()
    & (df["original_price"] > 0)
    & (df["price"] >= 0)
    & (df["price"] <= df["original_price"])
)

# Chỉ cập nhật các dòng có giá hợp lệ
df.loc[valid_price, "discount"] = (
    df.loc[valid_price, "original_price"]
    - df.loc[valid_price, "price"]
)

df.loc[valid_price, "discount_rate"] = (
    (
        df.loc[valid_price, "original_price"]
        - df.loc[valid_price, "price"]
    )
    / df.loc[valid_price, "original_price"]
    * 100
).round().astype(int)

# Không tự sửa các dòng có giá bất thường
print("Số dòng có giá không hợp lệ:", (~valid_price).sum())


# ==========================================
# 7. KIỂM TRA DỮ LIỆU ĐÁNH GIÁ
# ==========================================

if "rating_average" in df.columns:
    invalid_rating = ~df["rating_average"].between(
        0, 5, inclusive="both"
    )
    invalid_rating &= df["rating_average"].notna()

    print("Số rating ngoài phạm vi 0-5:", invalid_rating.sum())

if "review_count" in df.columns:
    invalid_review = df["review_count"] < 0
    print("Số review_count âm:", invalid_review.sum())


# ==========================================
# 8. XỬ LÝ GIÁ TRỊ THIẾU
# ==========================================

# Không tự điền 0 cho quantity_sold bị thiếu.
# Không xóa sản phẩm chỉ vì thiếu brand.
# Giữ NaN để tránh làm sai lệch dữ liệu.

print("\nSố giá trị thiếu sau xử lý:")
print(df.isna().sum()[df.isna().sum() > 0])
# ==========================================
# XÓA CỘT KHÔNG CẦN THIẾT
# ==========================================

drop_cols = ["sku", "inventory_status", "seller_id", "is_authentic"]

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns]
)

print("Đã xóa các cột:", drop_cols)


# ==========================================
# ĐỔI TÊN CỘT SANG TIẾNG VIỆT KHÔNG DẤU
# ==========================================

rename_cols = {
    "source": "Nguồn dữ liệu",
    "product_id": "Mã sản phẩm",
    "name": "Tên sản phẩm",
    "brand": "Thương hiệu",
    "seller_id": "Mã người bán",
    "seller_name": "Tên người bán",
    "price": "Giá bán",
    "original_price": "Giá gốc",
    "discount": "Số tiền giảm giá",
    "discount_rate": "Tỷ lệ giảm giá",
    "rating_average": "Điểm đánh giá trung bình",
    "review_count": "Số lượt đánh giá",
    "quantity_sold": "Số lượng đã bán",
    "order_count": "Số lượng đơn hàng",
    "favourite_count": "Số lượt yêu thích",
    "availability": "Trạng thái có sẵn",
    "is_authentic": "Xác thực chính hãng",
    "has_installment": "Có hỗ trợ trả góp",
    "has_gift": "Có quà tặng",
    "category_id": "Mã danh mục",
    "category_name": "Tên danh mục",
    "category_path": "Đường dẫn danh mục",
    "url": "Đường dẫn sản phẩm",
    "thumbnail_url": "Đường dẫn ảnh sản phẩm",
    "crawled_at": "Thời gian thu thập"
}

df = df.rename(columns=rename_cols)

print("\nCác cột sau khi chuẩn hóa:")
print(df.columns.tolist())

# Chuẩn hóa kiểu dữ liệu số lượng đã bán
df["Số lượng đã bán"] = pd.to_numeric(
    df["Số lượng đã bán"],
    errors="coerce"
)

# Kiểm tra dữ liệu
print("Kiểu dữ liệu:", df["Số lượng đã bán"].dtype)
print("Số giá trị thiếu:", df["Số lượng đã bán"].isna().sum())
print("Số giá trị âm:", (df["Số lượng đã bán"] < 0).sum())

# Giữ nguyên các giá trị thiếu, không tự động điền 0

# Chuẩn hóa tên danh mục sang tiếng Việt
category_mapping = {
    "dien-thoai-smartphone": "Điện thoại thông minh",
    "dinh-duong-cho-be": "Dinh dưỡng cho bé",
    "do-dung-nha-bep": "Đồ dùng nhà bếp",
    "gia-vi-va-che-bien": "Gia vị và chế biến",
    "cham-soc-toc-da-dau": "Chăm sóc tóc và da đầu",
    "nha-sach-tiki": "Nhà sách Tiki",
    "do-uong": "Đồ uống"
}
df["Tên danh mục"] = (
    df["Tên danh mục"]
    .astype("string")
    .str.strip()
    .replace(category_mapping)
)

# Kiểm tra kết quả
print(df["Tên danh mục"].value_counts(dropna=False))

# ==========================================
# 9. LƯU FILE ĐÃ LÀM SẠCH
# ==========================================

OUTPUT_FILE = OUTPUT_DIR / "tiki_cleaned.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN TẤT LÀM SẠCH TIKI")
print("Kích thước cuối cùng:", df.shape)
print("File đã lưu tại:", OUTPUT_FILE)

Kích thước dữ liệu ban đầu: (3074, 27)
Số lượng bị thiếu: 510

Kiểu dữ liệu:
float64


,product_id,name,price,quantity_sold,rating_average,review_count,url
0,279657517,Điện Thoại Xiaomi Redmi Note 15 Pro+ 5G 12GB/2...,10779000,NaN,0.0,0,https://tiki.vn/dien-thoai-xiaomi-redmi-note-1...
1,279419103,Điện Thoại Xiaomi Redmi A7 Pro 4GB/128GB - Hàn...,3989000,NaN,0.0,0,https://tiki.vn/dien-thoai-xiaomi-redmi-a7-pro...
2,279419089,Điện Thoại Xiaomi Redmi Note 15 Pro 8GB/256GB ...,7649000,NaN,0.0,0,https://tiki.vn/dien-thoai-xiaomi-redmi-note-1...
3,279418684,Điện Thoại Xiaomi Redmi 15 5G 4GB/128GB - Hàng...,4589000,NaN,0.0,0,https://tiki.vn/dien-thoai-xiaomi-redmi-15-5g-...
4,279418598,Điện Thoại Samsung Galaxy A07 5G 4GB/128GB - H...,4789000,NaN,0.0,0,https://tiki.vn/dien-thoai-samsung-galaxy-a07-...
5,279402741,Điện thoại Samsung Galaxy A07 5G (4GB/64GB) - ...,4690000,NaN,0.0,0,https://tiki.vn/dien-thoai-samsung-galaxy-a07-...
6,279389858,Điện thoại Xiaomi Redmi A7 Pro (4GB/64GB) - Hà...,4190000,NaN,0.0,0,https://tiki.vn/dien-thoai-xiaomi-redmi-a7-pro...
9,279303104,Điện Thoại Realme C85 5G 8GB/128GB - Hàng Chín...,6499000,NaN,0.0,0,https://tiki.vn/dien-thoai-realme-c85-5g-8gb-1...
10,279297038,Điện Thoại Samsung Galaxy A57 5G 8GB/128GB - H...,10799000,NaN,0.0,0,https://tiki.vn/dien-thoai-samsung-galaxy-a57-...
11,279277488,Điện thoại Samsung Galaxy A57 5G (8GB/128GB) -...,12490000,NaN,5.0,1,https://tiki.vn/dien-thoai-samsung-galaxy-a57-...



Một số giá trị quantity_sold:
quantity_sold
NaN     510
1.0     184
2.0     137
3.0      88
4.0      85
5.0      60
6.0      55
8.0      51
7.0      50
10.0     47
9.0      45
12.0     44
13.0     33
16.0     28
21.0     27
15.0     27
11.0     24
23.0     19
30.0     18
22.0     18
Name: count, dtype: int64
Đã chuẩn hóa tên cột.
Các cột đã loại bỏ: ['seller_name', 'inventory_status', 'order_count', 'favourite_count', 'has_installment', 'has_gift', 'availability']
Số cột còn lại: 20
Số dòng trùng hoàn toàn đã xóa: 0
Số product_id trùng: 0
Số SKU trùng: 0
Số dòng có giá không hợp lệ: 0
Số rating ngoài phạm vi 0-5: 0
Số review_count âm: 0

Số giá trị thiếu sau xử lý:
brand            496
quantity_sold    510
dtype: int64
Đã xóa các cột: ['sku', 'inventory_status', 'seller_id', 'is_authentic']

Các cột sau khi chuẩn hóa:
['Nguồn dữ liệu', 'Mã sản phẩm', 'Tên sản phẩm', 'Thương hiệu', 'Giá bán', 'Giá gốc', 'Số tiền giảm giá', 'Tỷ lệ giảm giá', 'Điểm đánh giá trung bình', 'Số lượt đánh giá

C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_13416\535778355.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object"]).columns


In [ ]:

import pandas as pd
from pathlib import Path

# Đường dẫn file Tiki đã làm sạch
FILE_PATH = Path("../../dataset/cleaned/tiki_cleaned.csv")

# Kiểm tra file tồn tại
if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {FILE_PATH.resolve()}"
    )

# Đọc dữ liệu
df_tiki = pd.read_csv(FILE_PATH)

# Kiểm tra cột thương hiệu
if "Thương hiệu" not in df_tiki.columns:
    raise ValueError(
        "Không tìm thấy cột 'Thương hiệu'. "
        f"Các cột hiện có: {df_tiki.columns.tolist()}"
    )

# Đếm số ô thiếu trước khi xử lý
missing_before = df_tiki["Thương hiệu"].isna().sum()

# Thay các ô thiếu bằng "Chưa xác định"
df_tiki["Thương hiệu"] = (
    df_tiki["Thương hiệu"].fillna("Chưa xác định")
)

# Lưu lại file Tiki
df_tiki.to_csv(
    FILE_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Kiểm tra kết quả
print("Số sản phẩm:", len(df_tiki))
print("Số ô thiếu thương hiệu trước xử lý:", missing_before)
print("Số ô thiếu thương hiệu sau xử lý:",
      df_tiki["Thương hiệu"].isna().sum())
print("\nCác giá trị thương hiệu sau xử lý:")
print(df_tiki["Thương hiệu"].value_counts().head(10))

print("\nĐã cập nhật file:", FILE_PATH.resolve())

display(df_tiki.head())


ValueError: Không tìm thấy cột 'thuong_hieu'. Các cột hiện có: ['Nguồn dữ liệu', 'Mã sản phẩm', 'Tên sản phẩm', 'Thương hiệu', 'Giá bán', 'Giá gốc', 'Số tiền giảm giá', 'Tỷ lệ giảm giá', 'Điểm đánh giá trung bình', 'Số lượt đánh giá', 'Số lượng đã bán', 'Mã danh mục', 'Tên danh mục', 'Đường dẫn danh mục', 'Đường dẫn sản phẩm', 'Đường dẫn ảnh sản phẩm', 'Thời gian thu thập']